以 tile_109_621.pt 文件为例，它是一个序列化的 PyTorch 字典，代表从全局空间网格坐标 (y=109, x=621) 处裁剪出的 512x512 局部图块。字典内包含四个键：input_expr 和 target_expr 分别是形状为 [512, 512, C] 的输入（核级别）与目标（细胞级别）基因表达矩阵；input_nuclei 和 target_cell_id 分别是形状为 [512, 512, 1] 的细胞核与细胞实例分割掩码（数值大于0代表不同实例的ID，背景0已被剔除）。为了最大化内存和 I/O 效率，这四个张量全部采用 torch.sparse_coo_tensor（稀疏 COO 格式）存储，仅保留非零元素，可直接被 PyTorch DataLoader 读取并用于深度学习模型的训练。
具体的处理代码在下方的generate_sparse_tiles函数。

In [ ]:
import scanpy as sc
import numpy as np

# 1. Load data
adata = sc.read_h5ad('/root/autodl-tmp/PRAD_sys_2um.h5ad')
cdata = sc.read_h5ad('/root/autodl-tmp/1_Processed_PRAD_sys_2um.h5ad')


libgomp: Invalid value for environment variable OMP_NUM_THREADS: 0


In [ ]:
adata.var_names_make_unique()
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.filter_cells(adata, min_counts=1)
adata

In [ ]:
# 2. Check Spatial Alignment
spatial_match = np.array_equal(adata.obsm['spatial'], cdata.obsm['spatial'])
print(f"Spatial coordinates match exactly: {spatial_match}")

# 3. Check Gene Channels
common_genes = np.intersect1d(adata.var_names, cdata.var_names)
print(f"adata genes (Target C): {adata.n_vars}")
print(f"cdata genes (Input C): {cdata.n_vars}")
print(f"Overlapping genes: {len(common_genes)}")

# 4. Check Mask Labels
print("\nTarget Cell Mask (adata.obs['cell_id']) sample:")
print(adata.obs['cell_id'].head(3))

print("\nInput Nucleus Mask (cdata.obs['nucleus_id']) sample:")
print(cdata.obs['nucleus_id'].head(3))

# 5. Calculate Global H and W
coords = adata.obsm['spatial']
x_max, y_max = coords.max(axis=0)
x_min, y_min = coords.min(axis=0)

W = int(x_max + 1)
H = int(y_max + 1)
print(f"\nGlobal Grid Size: H={H}, W={W}")

Spatial coordinates match exactly: True
adata genes (Target C): 3055
cdata genes (Input C): 2986
Overlapping genes: 2986

Target Cell Mask (adata.obs['cell_id']) sample:
71_369     0
107_340    0
91_355     0
Name: cell_id, dtype: category
Categories (192521, object): ['0', 'aaaagkdm-1', 'aaaamcnn-1', 'aaaamjle-1', ..., 'oiogjhoo-1', 'oiogldij-1', 'oioglipl-1', 'oiohaagd-1']

Input Nucleus Mask (cdata.obs['nucleus_id']) sample:
71_369     0
107_340    0
91_355     0
Name: nucleus_id, dtype: category
Categories (183931, object): ['0', 'aaaamcnn-1', 'aaaamjle-1', 'aaabjaeg-1', ..., 'oioggaih-1', 'oiogjhoo-1', 'oiogldij-1', 'oiohaagd-1']

Global Grid Size: H=6461, W=11507


In [13]:
import os
import torch
import numpy as np
import scanpy as sc

def generate_sparse_tiles(adata, cdata, save_dir, bin_size=2.0, patch_size=512, overlap=12):
    os.makedirs(save_dir, exist_ok=True)
    
    # 1. Coordinate Scaling
    coords = np.round(adata.obsm['spatial'] / bin_size).astype(np.int32)
    x_coords, y_coords = coords[:, 0], coords[:, 1]
    W, H = x_coords.max() + 1, y_coords.max() + 1
    
    # 2. Gene Alignment
    common_genes = cdata.var_names
    adata = adata[:, common_genes]
    C = len(common_genes)
    
    # 3. Label Encoding (Masks)
    target_mask_vals = adata.obs['cell_id'].astype('category').cat.codes.values
    input_mask_vals = cdata.obs['nucleus_id'].astype('category').cat.codes.values
    
    # 4. Extract Sparse Components
    a_coo = adata.X.tocoo()
    c_coo = cdata.X.tocoo()
    
    a_y, a_x = y_coords[a_coo.row], x_coords[a_coo.row]
    a_c, a_v = a_coo.col, a_coo.data
    
    c_y, c_x = y_coords[c_coo.row], x_coords[c_coo.row]
    c_c, c_v = c_coo.col, c_coo.data
    
    # 5. Calculate Offsets for Centered Tiling
    stride = patch_size - overlap
    
    # 计算 X 和 Y 方向最多能放多少个完整的 Tile
    nx = (W - patch_size) // stride + 1
    ny = (H - patch_size) // stride + 1
    
    # 计算未被覆盖的余数
    rem_w = W - (patch_size + (nx - 1) * stride)
    rem_h = H - (patch_size + (ny - 1) * stride)
    
    # 均分余数作为起始偏移量
    offset_x = rem_w // 2
    offset_y = rem_h // 2
    
    # 6. Sparse Tiling
    valid_count = 0
    
    for y0 in range(offset_y, offset_y + ny * stride, stride):
        for x0 in range(offset_x, offset_x + nx * stride, stride):
            # --- Target Expression ---
            a_idx = (a_y >= y0) & (a_y < y0 + patch_size) & (a_x >= x0) & (a_x < x0 + patch_size)
            if not a_idx.any():
                continue # Skip empty tiles
                
            ty_a, tx_a, tc_a = a_y[a_idx] - y0, a_x[a_idx] - x0, a_c[a_idx]
            target_expr = torch.sparse_coo_tensor(
                indices=torch.tensor(np.vstack([ty_a, tx_a, tc_a]), dtype=torch.int64),
                values=torch.tensor(a_v[a_idx], dtype=torch.float32),
                size=(patch_size, patch_size, C)
            ).coalesce()
            
            # --- Input Expression ---
            c_idx = (c_y >= y0) & (c_y < y0 + patch_size) & (c_x >= x0) & (c_x < x0 + patch_size)
            ty_c, tx_c, tc_c = c_y[c_idx] - y0, c_x[c_idx] - x0, c_c[c_idx]
            input_expr = torch.sparse_coo_tensor(
                indices=torch.tensor(np.vstack([ty_c, tx_c, tc_c]), dtype=torch.int64),
                values=torch.tensor(c_v[c_idx], dtype=torch.float32),
                size=(patch_size, patch_size, C)
            ).coalesce()
            
            # --- Masks ---
            m_idx = (y_coords >= y0) & (y_coords < y0 + patch_size) & (x_coords >= x0) & (x_coords < x0 + patch_size)
            local_y = y_coords[m_idx] - y0
            local_x = x_coords[m_idx] - x0
            
            # Target Mask
            t_vals = target_mask_vals[m_idx]
            t_valid = t_vals > 0
            target_mask = torch.sparse_coo_tensor(
                indices=torch.tensor(np.vstack([local_y[t_valid], local_x[t_valid], np.zeros_like(local_y[t_valid])]), dtype=torch.int64),
                values=torch.tensor(t_vals[t_valid], dtype=torch.int32),
                size=(patch_size, patch_size, 1)
            ).coalesce()
            
            # Input Mask
            i_vals = input_mask_vals[m_idx]
            i_valid = i_vals > 0
            input_mask = torch.sparse_coo_tensor(
                indices=torch.tensor(np.vstack([local_y[i_valid], local_x[i_valid], np.zeros_like(local_y[i_valid])]), dtype=torch.int64),
                values=torch.tensor(i_vals[i_valid], dtype=torch.int32),
                size=(patch_size, patch_size, 1)
            ).coalesce()
            
            # Save
            tile_data = {
                "input_expr": input_expr,
                "input_nuclei": input_mask,
                "target_expr": target_expr,
                "target_cell_id": target_mask
            }
            torch.save(tile_data, os.path.join(save_dir, f"tile_{y0}_{x0}.pt"))
            valid_count += 1

    print(f"Processed {valid_count} valid sparse tiles.")

In [14]:
generate_sparse_tiles(adata, cdata, save_dir='/root/autodl-tmp/tiles_new_512', bin_size=2.0, patch_size=512, overlap=12)

Processed 51 valid sparse tiles.


In [15]:
import shutil

source_dir = "/root/autodl-tmp/tiles_new_512"
output_path = "/root/autodl-tmp/tiles_new_512"  # 不需要加 .zip 后缀，函数会自动添加

shutil.make_archive(output_path, 'zip', source_dir)

'/root/autodl-tmp/tiles_new_512.zip'